# 00 · Setup, Cell Ranger outputs and the AnnData object
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kur1sutaru/ISMB-LATAM/blob/main/notebooks/en/00_setup_and_cellranger.ipynb)

**From Cells to Clusters: An Introduction to Single-Cell RNA-seq Analysis in Python**  
ISCB-LATAM SolBio SPBBC 2026 · Lima, Peru · November 9, 2026  
Instructor: Dr. Cristal Villalba Silva (UT Health San Antonio) · [github.com/Kur1sutaru](https://github.com/Kur1sutaru)

⏱ **Time in the tutorial:** 9:20–9:40 (20 min)

**By the end of this notebook you will be able to:**
- Run code in Google Colab
- Explain what Cell Ranger does, from FASTQ files to a count matrix
- Recognize the files in a Cell Ranger `outs/` folder and which ones you need
- Load a count matrix into Python and navigate an `AnnData` object

> 🟢 **Core** sections are covered live. 🔎 **Extra** sections are for self-study after the tutorial.

## How to use Google Colab
- Each grey box is a **code cell**. Click it and press `Shift + Enter` to run it.
- Cells run **in order**. If something breaks, use *Runtime → Restart session and run all*.
- Free Colab gives you a CPU machine with ~12 GB RAM. That is enough for this tutorial; no GPU needed.
- Colab machines are temporary: files disappear when the session ends (unless you use Google Drive).

## 0. Setup 🟢
Run the cell below first (▶ or `Shift + Enter`). On Google Colab it installs the packages (~1 min);
on your own computer it assumes you already ran `pip install -r requirements.txt`.

**Optional:** set `USE_DRIVE = True` to save your results in Google Drive, so the next notebooks
can reuse them. If you leave it `False`, each notebook rebuilds what it needs automatically.

In [ ]:
import os, sys, urllib.request
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q "scanpy[leiden,scrublet]>=1.10"

USE_DRIVE = False  # True = keep results in Google Drive
WORKDIR = "sc_tutorial"
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/sc_tutorial"
os.makedirs(WORKDIR, exist_ok=True)

# Tutorial helper functions (download + checkpoints)
if os.path.exists("../../scripts/sc_tutorial.py"):          # running from a local clone
    sys.path.insert(0, os.path.abspath("../../scripts"))
elif not os.path.exists("sc_tutorial.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Kur1sutaru/ISMB-LATAM/main/scripts/sc_tutorial.py", "sc_tutorial.py")
import sc_tutorial as tut

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)
sc.settings.figdir = os.path.join(WORKDIR, "figures")
print("scanpy", sc.__version__, "| working directory:", os.path.abspath(WORKDIR))

## 1. From the sequencer to a count matrix: Cell Ranger 🟢

A 10x Genomics experiment gives you **FASTQ files**. Each read pair carries:
- **Read 1**: the **cell barcode** (which droplet/cell, 16 bp) + the **UMI** (which original molecule, 10–12 bp);
- **Read 2**: the cDNA sequence (which gene).

**Cell Ranger** (`cellranger count`) turns those reads into a **cells × genes count matrix**:

1. **Align** Read 2 to the reference genome/transcriptome (STAR inside Cell Ranger).
2. **Correct barcodes** against the whitelist of valid 10x barcodes.
3. **Count UMIs**: reads with the same barcode + UMI + gene are PCR copies of one molecule → counted once.
4. **Call cells**: separate droplets that contain a cell from empty droplets (only ambient RNA).

You do not run Cell Ranger in Colab (it needs a Linux server with ~64 GB RAM). A typical command is:

```bash
cellranger count --id=sample1 \
    --transcriptome=refdata-gex-GRCh38-2024-A \
    --fastqs=/path/to/fastqs --sample=sample1 \
    --create-bam=true --localcores=16 --localmem=64
```

Open-source alternatives that produce the same kind of matrix: **STARsolo**, **alevin-fry / simpleaf**,
**kallisto | bustools**. They are often faster and are free to run anywhere.

### What is inside `outs/`?

| File | What it is | Do you need it? |
|---|---|---|
| `web_summary.html` | Interactive QC report | **Always open it first** |
| `metrics_summary.csv` | The same metrics as a table | Yes, to compare samples |
| `filtered_feature_bc_matrix/` (`barcodes.tsv.gz`, `features.tsv.gz`, `matrix.mtx.gz`) | Counts for **called cells** only | **Yes: this is what we analyze** |
| `filtered_feature_bc_matrix.h5` | Same matrix in a single HDF5 file | Yes (easiest to load) |
| `raw_feature_bc_matrix/` | Counts for **all** barcodes, including empty droplets | For ambient RNA removal (SoupX, CellBender) |
| `possorted_genome_bam.bam` | Aligned reads | Rarely (RNA velocity, genotyping) |
| `molecule_info.h5` | Per-molecule info | To aggregate/downsample samples |
| `cloupe.cloupe` | File for the Loupe Browser app | Quick visual exploration |

**Metrics to check in `web_summary.html`:** estimated number of cells, median genes per cell,
fraction of reads in cells (> 70 %), reads mapped confidently to the transcriptome (> 60 %),
sequencing saturation, and the shape of the **barcode rank ("knee") plot**.

## 2. Download a real Cell Ranger output 🟢
We use **PBMC 3k**: ~2,700 peripheral blood mononuclear cells from a healthy donor, sequenced by 10x Genomics
(public, CC BY 4.0). It is the "hello world" of single-cell analysis, so you can compare your results
with the Scanpy and Seurat tutorials. This dataset was made with an old Cell Ranger version, so the files
are called `genes.tsv` instead of `features.tsv.gz`, but the structure is the same.

In [ ]:
import tarfile
from pathlib import Path

data_dir = Path(WORKDIR) / "data"
tar_path = tut.download(tut.URLS["pbmc3k"], data_dir / "pbmc3k_filtered_gene_bc_matrices.tar.gz")
with tarfile.open(tar_path) as fh:
    fh.extractall(data_dir)

mtx_dir = data_dir / "filtered_gene_bc_matrices" / "hg19"
for f in sorted(mtx_dir.iterdir()):
    print(f"{f.name:15s} {f.stat().st_size / 1e6:6.1f} MB")

Let's peek at the three files. `barcodes.tsv` = one cell per line; `genes.tsv` = Ensembl ID + gene symbol;
`matrix.mtx` = a **sparse** matrix stored as *(gene, cell, count)* triplets, because >90 % of the values are zero.

In [ ]:
for name in ["barcodes.tsv", "genes.tsv", "matrix.mtx"]:
    print(f"--- {name} ---")
    with open(mtx_dir / name) as fh:
        for _ in range(4):
            print(fh.readline().rstrip())

## 3. Load the matrix into an `AnnData` object 🟢
`sc.read_10x_mtx()` reads the folder. For newer Cell Ranger outputs you can also use
`sc.read_10x_h5("filtered_feature_bc_matrix.h5")`.

In [ ]:
adata = sc.read_10x_mtx(mtx_dir, var_names="gene_symbols", cache=False)
adata.var_names_make_unique()  # a few gene symbols are duplicated
adata

### The anatomy of `AnnData`

```
                 var  (genes: gene_ids, highly_variable, ...)
              ┌───────────────────────────┐
  obs         │                           │   obsm: X_pca, X_umap (one row per cell)
 (cells:      │   X  (cells × genes)      │   layers: counts, ... (same shape as X)
  n_genes,    │                           │   uns: anything else (colors, parameters)
  cluster...) │                           │   obsp: cell–cell graphs (neighbors)
              └───────────────────────────┘
```
- `adata.X` → the matrix (here raw UMI counts, sparse)
- `adata.obs` → a pandas DataFrame with one row **per cell** (QC metrics, clusters, cell types...)
- `adata.var` → a pandas DataFrame with one row **per gene**
- `adata.obsm`, `adata.layers`, `adata.uns`, `adata.obsp` → embeddings, alternative matrices, metadata, graphs

The R equivalent is a **Seurat** or **SingleCellExperiment** object. We compare them in notebook 05.

In [ ]:
print("cells x genes:", adata.shape)
print("first barcodes:", adata.obs_names[:3].tolist())
print("first genes:   ", adata.var_names[:3].tolist())
print("matrix type:   ", type(adata.X).__name__)
sparsity = 1 - adata.X.nnz / (adata.n_obs * adata.n_vars)
print(f"zeros in the matrix: {sparsity:.1%}")
adata.var.head()

### Basic numbers per cell
Total UMIs per cell (**library size**) and number of detected genes are the first things to look at.

In [ ]:
adata.obs["total_counts"] = np.asarray(adata.X.sum(axis=1)).ravel()
adata.obs["n_genes"] = np.asarray((adata.X > 0).sum(axis=1)).ravel()
adata.obs.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(adata.obs["total_counts"], bins=60, color="#2a9d8f")
axes[0].set(xlabel="UMIs per cell", ylabel="cells")
axes[1].hist(adata.obs["n_genes"], bins=60, color="#e76f51")
axes[1].set(xlabel="genes per cell")
# Barcode rank ("knee") plot, like in web_summary.html
ranked = np.sort(adata.obs["total_counts"].values)[::-1]
axes[2].loglog(np.arange(1, len(ranked) + 1), ranked, color="#264653")
axes[2].set(xlabel="barcode rank", ylabel="UMIs")
plt.tight_layout()

> The knee plot here only shows **called cells** (filtered matrix). In the `raw_feature_bc_matrix` you would
> also see the long tail of empty droplets after the knee.

### Which genes take most of the reads?

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

Mitochondrial (`MT-`), ribosomal (`RPS`/`RPL`) and very abundant genes like `MALAT1` dominate.
This is normal, and it is why we need quality control and normalization (next notebook).

### 💡 Your turn
1. How many UMIs does the **first** cell have? 2. Which gene has the highest **total** count across all cells?
*Hint:* `adata.X.sum(axis=0)` sums over cells.

In [ ]:
# Your code here

<details><summary><b>Show solution</b></summary>

```python
print(adata.obs["total_counts"].iloc[0])
gene_totals = np.asarray(adata.X.sum(axis=0)).ravel()
print(adata.var_names[gene_totals.argmax()])
```
</details>

## 4. 🔎 Extra: the same in R (Seurat)
```r
library(Seurat)
counts <- Read10X("filtered_gene_bc_matrices/hg19/")   # or Read10X_h5("filtered_feature_bc_matrix.h5")
pbmc <- CreateSeuratObject(counts, project = "pbmc3k", min.cells = 3, min.features = 200)
pbmc
```
Converting between ecosystems: save `adata.write_h5ad("pbmc.h5ad")` and read it in R with
`zellkonverter::readH5AD()` (SingleCellExperiment) or `schard::h5ad2seurat()`.

## 5. 🔎 Extra: analyzing your own data
Upload your `filtered_feature_bc_matrix.h5` (folder icon on the left → upload) or copy it to Google Drive, then:
```python
adata = sc.read_10x_h5("filtered_feature_bc_matrix.h5")
adata.var_names_make_unique()
```
For several samples, read each one, add `adata.obs["sample"] = "name"` and join them with
`anndata.concat([...], label="sample")`.

---
## ✅ Done!
You know where the count matrix comes from and how it lives in Python. Next: **01 · Quality control and preprocessing**.